# 01 · Seleção de variáveis e clusterização
Leia `data/terra/terra.parquet` e os insumos em `data/terra/spec_analogos/`, os nichos em `config/` e os perfis fixos em `data/lua/` e `data/marte/`.
A ordem das variáveis no JSON define a prioridade na seleção. A previsão considera as variáveis dinâmicas dos clusters e dos nichos.


## Configurar caminhos

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/feynmans_legacy/analog_study.py").exists()
)
sys.path.insert(0, str(ROOT / 'src'))
TERRA = ROOT / 'data/terra'
DATA = TERRA / 'spec_analogos'
RUN = TERRA / 'analise_analogos'
RUN.mkdir(parents=True, exist_ok=True)

## Conferir os arquivos de entrada

In [2]:
import json

from pyarrow.parquet import ParquetFile

from feynmans_legacy.analog_study import (
    KEY,
    audit_profile_catalog,
    cluster_snapshot,
    months,
    profile_criteria,
    select_features,
    validate_key,
)

required = [
    TERRA / "terra.parquet",
    *(DATA / name for name in (
        "estaticas.parquet", "coordenadas.parquet", "dicionario_spec.csv",
    )),
    ROOT / "config/nichos.json",
    ROOT / "config/criterios_referencia_proposta.csv",
    ROOT / "data/lua/lua.parquet",
    ROOT / "data/marte/marte.parquet",
]
missing = [str(path.relative_to(ROOT)) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Arquivos ausentes: {', '.join(missing)}")

## Carregar os Parquets e validar as chaves

In [3]:
terra_path = TERRA / "terra.parquet"
keys = pd.read_parquet(terra_path, columns=KEY)
static = pd.read_parquet(DATA / "estaticas.parquet")
coords = pd.read_parquet(DATA / "coordenadas.parquet")

validate_key(keys, KEY, "terra")
validate_key(static, ["location_id"], "estaticas")
validate_key(coords, ["location_id"], "coordenadas")
if keys.empty or not pd.api.types.is_integer_dtype(keys["anomes"]):
    raise ValueError("terra.parquet precisa conter anomes inteiro no formato YYYYMM.")
observed_months = keys["anomes"].unique()
if not pd.Series(observed_months).astype(str).str.fullmatch(r"[0-9]{6}").all():
    raise ValueError("anomes precisa ter seis dígitos no formato YYYYMM.")
months(pd.Series(observed_months))
loc_ids = set(keys["location_id"].unique())
if loc_ids - set(coords["location_id"]) or loc_ids - set(static["location_id"]):
    raise ValueError("Há IDs da série mensal ausentes no de-para ou na tabela estática.")
if not (coords.latitude.between(-90, 90) & coords.longitude.between(-180, 180)).all():
    raise ValueError("Coordenadas ausentes ou fora dos limites geográficos.")
print(f"{len(keys):,} observações em {len(loc_ids):,} locais")
del keys

7,886,792 observações em 85,726 locais


## Identificar as variáveis dos nichos

In [4]:
niches_path = ROOT / "config/nichos.json"
niches = json.loads(niches_path.read_text(encoding="utf-8"))
candidates = list(dict.fromkeys(
    item["terra"] for body in niches.values() for niche in body.values() for item in niche
))
dynamic = set(ParquetFile(terra_path).schema_arrow.names) - set(KEY)
static_columns = set(static.columns) - {"location_id"}
if dynamic & static_columns:
    repeated = sorted(dynamic & static_columns)
    raise ValueError(f"Colunas duplicadas entre as specs terrestres: {repeated}")
missing_columns = set(candidates) - (dynamic | static_columns)
if missing_columns:
    raise ValueError(f"Variáveis dos nichos ausentes nas specs: {sorted(missing_columns)}")
references = {}
for body in ("lua", "marte"):
    reference = pd.read_parquet(ROOT / "data" / body / f"{body}.parquet")
    if reference.empty:
        raise ValueError(f"Spec de {body} sem bases cadastradas.")
    references[body] = reference
    for base_id in reference["base_id"]:
        profile_criteria(niches_path, body, "Todos", reference, base_id)

reference_month = int(max(observed_months))
dynamic_candidates = [name for name in candidates if name in dynamic]
snapshot = pd.read_parquet(
    terra_path, columns=[*KEY, *dynamic_candidates], filters=[("anomes", "==", reference_month)]
).merge(static, on="location_id", how="left", validate="one_to_one")
catalog = pd.read_csv(ROOT / "config/criterios_referencia_proposta.csv")
dictionary = pd.read_csv(DATA / "dicionario_spec.csv")
audit_profile_catalog(niches, references, snapshot, catalog, dictionary)
print(f"{len(catalog)} critérios compatíveis com a configuração e o dicionário terrestre")


222 critérios compatíveis com a configuração e o dicionário terrestre


## Selecionar variáveis e agrupar os locais

In [5]:
chosen, report = select_features(snapshot, candidates)
clusters = cluster_snapshot(snapshot, chosen, n_clusters=3)
targets = [name for name in dict.fromkeys(chosen + candidates) if name in dynamic]

## Salvar os resultados

In [6]:
for name, frame in {
    "estaticas": static, "coordenadas": coords, "clusters": clusters, "selecao": report,
}.items():
    frame.to_parquet(RUN / f"{name}.parquet", index=False)
(RUN / "selecao.json").write_text(json.dumps({
    "anomes_cluster": reference_month,
    "anomes_observados": sorted(map(int, observed_months)),
    "variaveis_cluster": chosen,
    "alvos_previsao": targets,
}, indent=2), encoding="utf-8")

2617

## Conferir a seleção e os clusters

In [7]:
print(report.to_string(index=False))
print(snapshot.join(clusters["cluster"]).groupby("cluster")[chosen].mean())

                                             variavel  cobertura                                                              decisao
                                 temperatura__ar_2m_c   1.000000                                                          selecionada
               temperatura__lst_amplitude_dia_noite_c   0.994424                                                          selecionada
        iluminacao_radiacao__elevacao_solar_max_graus   1.000000                                                          selecionada
                  iluminacao_radiacao__dias_dia_polar   1.000000                                  correlacao com temperatura__ar_2m_c
   iluminacao_radiacao__radiacao_solar_superficie_wm2   0.697770                                               cobertura insuficiente
                         agua_gelo__indice_permafrost   1.000000                                                          selecionada
                       agua_gelo__precipitacao_mm_dia   1.0000